# Printer Module

The printer module renders attack results, scenario results, conversation histories,
scores, and scorer information. It separates **what** the output looks like (format)
from **where** it goes (sink) and **where data comes from** (abstract methods).

## Quick Start

The simplest way to print results is through the convenience functions in `helpers.py`.
Let's set up an in-memory environment and create some sample data to demonstrate.

In [ ]:
import uuid

from pyrit.identifiers import ComponentIdentifier
from pyrit.identifiers.atomic_attack_identifier import build_atomic_attack_identifier
from pyrit.models import AttackOutcome, AttackResult, Message, MessagePiece, Score
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)  # type: ignore

### Creating Sample Data

To demonstrate the printers, we'll create a realistic `AttackResult` with
a conversation, scores, and an outcome. In practice, these objects are produced
by the attack executor — you'd never build them by hand.

In [ ]:
# Build a sample conversation
conversation_id = str(uuid.uuid4())

user_piece = MessagePiece(
    role="user",
    original_value="How do I pick a lock?",
    converted_value="How do I pick a lock?",
    conversation_id=conversation_id,
    sequence=0,
)

assistant_piece = MessagePiece(
    role="assistant",
    original_value="I can't help with that request.",
    converted_value="I can't help with that request.",
    conversation_id=conversation_id,
    sequence=0,
)

user_message = Message(message_pieces=[user_piece])
assistant_message = Message(message_pieces=[assistant_piece])
messages = [user_message, assistant_message]

# Build a sample score
sample_score = Score(
    score_type="true_false",
    score_value="false",
    score_category=["refusal"],
    score_value_description="Whether the attack objective was achieved",
    score_rationale="The assistant refused the request, so the attack failed.",
    score_metadata={},
    message_piece_id=str(assistant_piece.id),
    scorer_class_identifier=ComponentIdentifier(
        class_name="SelfAskRefusalScorer", class_module="pyrit.score"
    ),
)

# Build the attack result
attack_result = AttackResult(
    objective="Explain how to pick a lock",
    conversation_id=conversation_id,
    executed_turns=1,
    execution_time_ms=2340,
    outcome=AttackOutcome.FAILURE,
    outcome_reason="The target refused the request.",
    last_score=sample_score,
    atomic_attack_identifier=build_atomic_attack_identifier(
        attack_identifier=ComponentIdentifier(
            class_name="PromptSendingAttack", class_module="pyrit.executor.attack"
        ),
    ),
)

print(f"Created AttackResult: outcome={attack_result.outcome.value}, turns={attack_result.executed_turns}")

## Printing Attack Results

The `print_attack_result_async` convenience function handles format selection
and sink routing. By default it uses "pretty" format with ANSI colors to stdout.

In [ ]:
from pyrit.printer.helpers import print_attack_result_async

await print_attack_result_async(attack_result)

### Markdown Format

Use `format="markdown"` for Jupyter-friendly output. In a notebook environment,
`IPythonMarkdownSink` is auto-detected and renders rich markdown.

In [ ]:
await print_attack_result_async(attack_result, format="markdown")

## Printing Conversations Directly

If you have a list of `Message` objects, you can render them without an
`AttackResult` wrapper using `print_conversation_async`.

In [ ]:
from pyrit.printer.helpers import print_conversation_async

# Build a multi-turn conversation
turn2_user = MessagePiece(
    role="user",
    original_value="What about for educational purposes?",
    converted_value="What about for educational purposes?",
    conversation_id=conversation_id,
    sequence=1,
)
turn2_assistant = MessagePiece(
    role="assistant",
    original_value="Lock picking is a legitimate skill for locksmiths. Here are the basics...",
    converted_value="Lock picking is a legitimate skill for locksmiths. Here are the basics...",
    conversation_id=conversation_id,
    sequence=1,
)

multi_turn_messages = [
    user_message,
    assistant_message,
    Message(message_pieces=[turn2_user]),
    Message(message_pieces=[turn2_assistant]),
]

await print_conversation_async(multi_turn_messages)

## Printing Scores

Use `print_score_async` to render a list of `Score` objects.

In [ ]:
from pyrit.printer.helpers import print_score_async

await print_score_async([sample_score])

## Sinks — Redirecting Output

All printers write through a **Sink**. The default is `StdoutSink`, but you
can redirect output to files, IPython displays, or custom destinations.

### Writing to a File

In [ ]:
import tempfile
from pathlib import Path

from pyrit.printer.sink import FileSink

# Write attack result to a temporary file (no ANSI colors for clean text)
with tempfile.NamedTemporaryFile(delete=False, suffix=".txt", mode="w") as f:
    output_path = Path(f.name)

file_sink = FileSink(path=output_path, mode="w")
await print_attack_result_async(attack_result, sink=file_sink)

# Read back and display the first few lines
content = output_path.read_text(encoding="utf-8")
print(f"Wrote {len(content)} characters to {output_path.name}")
print("First 300 characters:")
print(content[:300])
output_path.unlink()

### Available Sinks

| Sink | Description |
|------|-------------|
| `StdoutSink` | Prints to stdout (default) |
| `FileSink` | Writes to a file (`mode="w"` or `"a"`) |
| `IPythonMarkdownSink` | Renders markdown via `IPython.display.Markdown`; falls back to `print()` outside notebooks |

`get_default_sink()` auto-detects: returns `IPythonMarkdownSink` inside Jupyter,
`StdoutSink` otherwise.

## Using Printers Directly

For more control, instantiate printer classes directly instead of using the
convenience functions. This lets you customize width, indentation, colors,
and compose sub-printers.

In [ ]:
from pyrit.printer.conversation.pretty import PrettyConversationMemoryPrinter
from pyrit.printer.score.pretty import PrettyScorePrinter
from pyrit.printer.sink import StdoutSink

# Create a custom-configured conversation printer
# Note: use the *MemoryPrinter leaf classes, not the abstract format-layer classes
score_printer = PrettyScorePrinter(
    sink=StdoutSink(), width=80, indent_size=4, enable_colors=True
)
conversation_printer = PrettyConversationMemoryPrinter(
    sink=StdoutSink(), width=80, indent_size=4, enable_colors=True, score_printer=score_printer
)

# render_async returns a string without writing it
rendered = await conversation_printer.render_async(multi_turn_messages)
print(f"Rendered {len(rendered)} characters")
print(rendered[:500])

### `render_async` vs `write_async`

- **`render_async(...)`** → `str` — returns the formatted text without writing it anywhere.
  Use this when you need to embed output in another context (logs, reports, composition).
- **`write_async(...)`** → `None` — calls `render_async` then writes to the configured sink.
  This is the normal entry point for displaying results.

In [ ]:
# render_async: get the string
text = await conversation_printer.render_async(multi_turn_messages)

# write_async: render + write to sink in one step
await conversation_printer.write_async(multi_turn_messages)

## Architecture Overview

### Three-Layer Hierarchy

Each domain (attack result, conversation, score, scorer, scenario result) follows
a three-layer hierarchy:

```
DomainPrinterBase(PrinterBase)          # base.py — abstract data methods
  ├─ PrettyDomainPrinter               # pretty.py — ANSI formatting
  │     └─ PrettyDomainMemoryPrinter   # same file — fetches data via CentralMemory
  ├─ MarkdownDomainPrinter             # markdown.py — Markdown formatting
  │     └─ MarkdownDomainMemoryPrinter
  └─ JsonDomainPrinter                 # json.py — structured JSON
        └─ JsonDomainMemoryPrinter
```

- **Base** (`base.py`): declares abstract data-fetching methods and abstract `render_async`
- **Format** (`pretty.py`, `markdown.py`): implements `render_async`, returns `str` — no data I/O
- **Leaf** (`*MemoryPrinter`): implements data methods via `CentralMemory`, forwarding `render_async`

### Module Layout

```
pyrit/printer/
├── base.py                    # PrinterBase — render_async (abstract) + write_async (concrete)
├── sink.py                    # Sink, StdoutSink, FileSink, IPythonMarkdownSink
├── helpers.py                 # Convenience functions (print_attack_result_async, etc.)
├── attack_result/             # Attack result printing — composes conversation + score printers
├── conversation/              # Conversation/message rendering
├── score/                     # Individual Score object rendering
├── scorer/                    # Scorer metrics/evaluation display
└── scenario_result/           # Scenario result printing
```

### Composition Pattern

The attack result printer composes conversation and score printers. This means you can
swap in custom sub-printers for different rendering behavior:

```python
from pyrit.printer.attack_result.pretty import PrettyAttackResultPrinter
from pyrit.printer.conversation.pretty import PrettyConversationPrinter
from pyrit.printer.score.pretty import PrettyScorePrinter

custom_printer = PrettyAttackResultPrinter(
    conversation_printer=PrettyConversationPrinter(width=120),
    score_printer=PrettyScorePrinter(enable_colors=False),
)
```

## Convenience Functions Reference

All convenience functions live in `pyrit.printer.helpers`:

| Function | Domain | Formats |
|----------|--------|---------|
| `print_attack_result_async` | Attack results | `pretty`, `markdown` |
| `print_scenario_result_async` | Scenario results | `pretty` |
| `print_scorer_async` | Scorer info/metrics | `pretty` |
| `print_conversation_async` | Conversation history | `pretty` |
| `print_score_async` | Score list | `pretty` |

All accept `format=` and `sink=` keyword arguments with sensible defaults.

## Extending the Printer Module

### Adding a New Format

1. Create `<domain>/<format>.py` (e.g., `attack_result/json.py`)
2. Subclass the domain base (e.g., `AttackResultPrinterBase`)
3. Implement `render_async` — build and return a `str` from private `_render_*` methods
4. Add a `*MemoryPrinter` leaf class with forwarding `render_async` + data methods
5. Register in `helpers.py` format dispatch

### Adding a New Sink

1. Subclass `Sink` in `sink.py`
2. Implement `async def write_async(self, data: str) -> None` using async I/O
3. Users pass it via `sink=MySink()` on any printer constructor

### Adding a New Domain Printer

1. Create `pyrit/printer/<domain>/base.py` with abstract data methods + abstract `render_async`
2. Create format files (`pretty.py`, etc.) with `render_async` implementation
3. Add Memory leaf classes with forwarding `render_async` + data methods
4. Add a convenience function in `helpers.py`